# Image Classification on MNIST using Recurrent Neural Networks (RNN, GRU, LSTM)

This notebook demonstrates how to process sequential data using recurrent neural architectures in PyTorch. We treat MNIST images (28x28 pixels) as a sequence of 28 rows, where each row has 28 features, and feed them into standard RNN, GRU, and LSTM architectures to perform digit classification.

### Repository
Designed for: [DeepLearning_with_PyTorch](https://github.com/ShivaUlavi/DeepLearning_with_PyTorch)

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

## 1. Setup & Environment Configurations

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
print(device)

cuda


## 2. Model Hyperparameters

In [ ]:
#HyperParameters
input_size = 28
sequence_length = 28
hidden_size = 128
num_layers = 3

num_classes = 10
num_epochs = 10
batch_size = 256
learning_rate = 0.001

## 3. Data Ingestion (MNIST Dataset)

In [ ]:
#MNIST Dataset

train_dataset = torchvision.datasets.MNIST(root= './data/',
                                          train = True,
                                          transform= transforms.ToTensor(),
                                          download=False) ##Set True to download

test_dataset = torchvision.datasets.MNIST(root= './data/',
                                          train = False,
                                          transform= transforms.ToTensor())


In [ ]:
#Data loader
train_loader = torch.utils.data.DataLoader(dataset = train_dataset,
                                          batch_size= batch_size,
                                          shuffle = True)

test_loader = torch.utils.data.DataLoader(dataset = test_dataset,
                                          batch_size= batch_size,
                                          shuffle = False)

## 4. Model Architectures
We implement three different sequential models:
- **Elman RNN**: Simple Recurrent Neural Network.
- **GRU**: Gated Recurrent Unit, designed to address vanishing gradient issues using update and reset gates.
- **LSTM**: Long Short-Term Memory, utilizing input, forget, output, and cell state gates.

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(RNN, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True) #x needs to be (batch_size, sequence, input_size)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        #set the initial hidden state h0
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        #x: (n, 28,28) h0: (3, n, 128)
        out, _ = self.rnn(x, h0)
        #out; (n, 28, 128) i.e. (batch_size, sequence_length, hidden_size)

        #decode the hidden size of the last time step
        out = out[:, -1, :] #out: (n, 128)

        out = self.fc(out) #out: (n, 10)
        return out

In [ ]:
class GRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(GRU, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first = True) #x needs to be (batch_size, sequence, input_size)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        #set the initial hidden state h0
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        #x: (n, 28,28) h0: (3, n, 128)
        out, _ = self.gru(x, h0)
        #out; (n, 28, 128) i.e. (batch_size, sequence_length, hidden_size)

        #decode the hidden size of the last time step
        out = out[:, -1, :] #out: (n, 128)

        out = self.fc(out) #out: (n, 10)
        return out

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTM, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first = True) #x needs to be (batch_size, sequence, input_size)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        #set the initial hidden state h0
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        #x: (n, 28,28) h0: (3, n, 128)
        out, _ = self.lstm(x, (h0,c0))
        #out; (n, 28, 128) i.e. (batch_size, sequence_length, hidden_size)

        #decode the hidden size of the last time step
        out = out[:, -1, :] #out: (n, 128)

        out = self.fc(out) #out: (n, 10)
        return out

## 5. Training and Evaluation Routines

In [ ]:
def training_step(model,criterion, data, num_epochs):

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    n_total_steps = len(data)
    for epoch in range(num_epochs):
        for i, (images,labels) in enumerate(data):
            # origin shape: [N, 1, 28, 28]
            # resized: [N, 28, 28]
            images = images.reshape(-1, sequence_length, input_size).to(device)
            labels = labels.to(device)

            #forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            #backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i+1) % 100 == 0:
                print (f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{n_total_steps}], Loss: {loss.item():.4f}')



In [ ]:
def test_model(model, test_loader):
  model.eval()  # Put model in evaluation mode
  n_correct = 0
  n_samples = 0

  # In test phase, we don't need to compute gradients (for memory efficiency)
  with torch.no_grad():
    for images, labels in test_loader:
      images = images.reshape(-1, sequence_length, input_size).to(device)
      labels = labels.to(device)
      outputs = model(images)

      # max returns (value ,index)
      _, predicted = torch.max(outputs.data, 1)
      n_samples += labels.size(0)
      n_correct += (predicted == labels).sum().item()

  acc = 100.0 * n_correct / n_samples
  print(f"Accuracy of the network on the test images: {acc:.2f}%")
  return acc

## 6. Training and Evaluation Runs

In [ ]:
model_rnn = RNN(input_size, hidden_size, num_layers, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

training_step(model_rnn, criterion, data= train_loader,num_epochs=num_epochs)

Epoch [1/10], Step [100/235], Loss: 0.6842
Epoch [1/10], Step [200/235], Loss: 0.5168
Epoch [2/10], Step [100/235], Loss: 0.3552
Epoch [2/10], Step [200/235], Loss: 0.1921
Epoch [3/10], Step [100/235], Loss: 0.1927
Epoch [3/10], Step [200/235], Loss: 0.0890
Epoch [4/10], Step [100/235], Loss: 0.1910
Epoch [4/10], Step [200/235], Loss: 0.2013
Epoch [5/10], Step [100/235], Loss: 0.1739
Epoch [5/10], Step [200/235], Loss: 0.1351
Epoch [6/10], Step [100/235], Loss: 0.0956
Epoch [6/10], Step [200/235], Loss: 0.1316
Epoch [7/10], Step [100/235], Loss: 0.0925
Epoch [7/10], Step [200/235], Loss: 0.0982
Epoch [8/10], Step [100/235], Loss: 0.0594
Epoch [8/10], Step [200/235], Loss: 0.1425
Epoch [9/10], Step [100/235], Loss: 0.0561
Epoch [9/10], Step [200/235], Loss: 0.0639
Epoch [10/10], Step [100/235], Loss: 0.1374
Epoch [10/10], Step [200/235], Loss: 0.0289


In [ ]:
test_model(model_rnn, test_loader)

Accuracy of the network on the test images: 97.26%


97.26

In [ ]:
model_lstm = LSTM(input_size, hidden_size, num_layers, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

training_step(model_lstm, criterion, data= train_loader,num_epochs=num_epochs)

Epoch [1/10], Step [100/235], Loss: 0.5909
Epoch [1/10], Step [200/235], Loss: 0.2469
Epoch [2/10], Step [100/235], Loss: 0.1856
Epoch [2/10], Step [200/235], Loss: 0.1436
Epoch [3/10], Step [100/235], Loss: 0.1067
Epoch [3/10], Step [200/235], Loss: 0.1252
Epoch [4/10], Step [100/235], Loss: 0.1044
Epoch [4/10], Step [200/235], Loss: 0.0830
Epoch [5/10], Step [100/235], Loss: 0.0975
Epoch [5/10], Step [200/235], Loss: 0.0910
Epoch [6/10], Step [100/235], Loss: 0.1048
Epoch [6/10], Step [200/235], Loss: 0.0478
Epoch [7/10], Step [100/235], Loss: 0.0735
Epoch [7/10], Step [200/235], Loss: 0.0402
Epoch [8/10], Step [100/235], Loss: 0.0206
Epoch [8/10], Step [200/235], Loss: 0.0158
Epoch [9/10], Step [100/235], Loss: 0.0147
Epoch [9/10], Step [200/235], Loss: 0.0445
Epoch [10/10], Step [100/235], Loss: 0.0720
Epoch [10/10], Step [200/235], Loss: 0.0420


In [ ]:
test_model(model_lstm, test_loader)

Accuracy of the network on the test images: 98.61%


98.61

In [ ]:
model_gru = GRU(input_size, hidden_size, num_layers, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

training_step(model_gru, criterion, data= train_loader,num_epochs=num_epochs)

Epoch [1/10], Step [100/235], Loss: 0.5084
Epoch [1/10], Step [200/235], Loss: 0.2766
Epoch [2/10], Step [100/235], Loss: 0.1486
Epoch [2/10], Step [200/235], Loss: 0.1864
Epoch [3/10], Step [100/235], Loss: 0.1119
Epoch [3/10], Step [200/235], Loss: 0.0698
Epoch [4/10], Step [100/235], Loss: 0.0445
Epoch [4/10], Step [200/235], Loss: 0.0607
Epoch [5/10], Step [100/235], Loss: 0.0682
Epoch [5/10], Step [200/235], Loss: 0.0827
Epoch [6/10], Step [100/235], Loss: 0.0274
Epoch [6/10], Step [200/235], Loss: 0.0510
Epoch [7/10], Step [100/235], Loss: 0.0584
Epoch [7/10], Step [200/235], Loss: 0.0128
Epoch [8/10], Step [100/235], Loss: 0.0237
Epoch [8/10], Step [200/235], Loss: 0.0294
Epoch [9/10], Step [100/235], Loss: 0.0344
Epoch [9/10], Step [200/235], Loss: 0.0176
Epoch [10/10], Step [100/235], Loss: 0.0304
Epoch [10/10], Step [200/235], Loss: 0.0461


In [ ]:
test_model(model_gru, test_loader)

Accuracy of the network on the test images: 98.53%


98.53

## 7. Comparative Summary

Below is a summary of the performance of the three models on the MNIST test dataset using the same hyperparameters:
- **Learning Rate**: 0.001
- **Batch Size**: 256
- **Epochs**: 10
- **Sequence Length / Input Size**: 28 x 28

| Model Architecture | Final Training Loss | Test Accuracy |
| :--- | :---: | :---: |
| **Standard RNN** | 0.0289 | 97.26% |
| **GRU** | 0.0461 | 98.53% |
| **LSTM** | 0.0420 | **98.61%** |

### Key Takeaways
1. **Performance**: Both gated architectures (LSTM and GRU) outperformed the vanilla RNN. LSTM achieved the highest accuracy of **98.61%** closely followed by GRU at **98.53%**.
2. **Training Stability**: The loss decrease was more stable and smoother in GRU/LSTM models compared to vanilla RNN, showcasing the effectiveness of gating mechanisms in maintaining gradients over sequence steps.